# 오답 분석 (CPU 런타임 · 유닛 거의 0) — 남은 3.5% 는 어디에 있나

**전제**: dev 다수결 정답은 약 27% 가 틀린 노이즈 → dev 오답 분석은 의미 없음.
캐글 0.9645 의 남은 3.5%(약 240문항) 는 **진짜 모델 오답**일 가능성이 큼 → test 에서 찾음.

**방법**: 정답은 모르지만, 오답은 **우리 모델들이 서로 엇갈리거나 확신이 약한 문항**에 몰려 있음.
1. 모든 모델의 test 확률로 최고점 조합(m2) 재현 → 문항별 확신도 · 모델 간 일치 수
2. 불확실 문항의 **질문 유형 · 원본 이미지 크기** 를 확신 문항과 비교 → 무엇이 약점인지
3. 가장 불확실한 문항 사진을 직접 보기

런타임 유형 **CPU** 로 · 모두 실행

# 1. 모델 확률 · test 불러오기

In [ ]:
import unicodedata, zipfile, io, re, base64
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML
from google.colab import drive
drive.mount("/content/drive")
NFC = lambda s: unicodedata.normalize("NFC", s)
MY = Path("/content/drive/MyDrive")
def find(tag):
    hit = [p for p in MY.rglob("*.npy") if tag in NFC(p.name) and "ensemble_outputs" not in str(p)]
    return hit[0] if hit else None
norm = lambda a: a / a.sum(1, keepdims=True)
F = {"동현TTA": "동현_TTA_8B_640px", "수혁TTA4": "수혁_TTA_8B_640px", "B3": "수혁_B3_3B_px1024",
     "8B1024s": "동현_8B_1024px", "1024full": "동현_TTA_8B_1024px_full", "32B": "동현_32B_1024px"}
P = {}
for k, tag in F.items():
    f = find(tag)
    print(f"  {k:<9}", f.name if f else "없음")
    if f: P[k] = norm(np.load(f))

DZ = next(MY.rglob("data.zip"))
Z = zipfile.ZipFile(DZ)
pick = lambda fn: next(n for n in Z.namelist() if n.endswith(fn) and "__MACOSX" not in n)
test = pd.read_csv(io.BytesIO(Z.read(pick("test.csv"))))
NAMES = {n.replace("\\", "/").split("/", 1)[-1] if n.count("/") > 1 else n: n for n in Z.namelist()}
def member(rel):
    rel = str(rel).replace("\\", "/")
    for n in (rel, f"data/{rel}"):
        if n in Z.namelist(): return n
    return next(n for n in Z.namelist() if n.endswith(rel))
print("test", len(test), "| 모델", list(P))

# 2. 최고점 조합 재현 · 불확실 문항 찾기

In [ ]:
# 최고점 조합 m2 (0.96455) 재현: 동현TTA 18 : 수혁TTA4 18 : B3 9 : 8B1024s 15 : 32B 40
W = {"동현TTA": .18, "수혁TTA4": .18, "B3": .09, "8B1024s": .15, "32B": .40}
W = {k: v for k, v in W.items() if k in P}; s = sum(W.values())
ens = sum(v / s * P[k] for k, v in W.items())
L = np.array(list("abcd"))
test["pred"] = L[ens.argmax(1)]
srt = np.sort(ens, 1)
test["conf"] = srt[:, -1]; test["margin"] = srt[:, -1] - srt[:, -2]
votes = np.stack([P[k].argmax(1) for k in P], 1)
test["agree"] = (votes == ens.argmax(1)[:, None]).sum(1)          # 최종 답에 동의한 모델 수
NM = len(P)
print(f"모델 {NM}개 중 최종 답에 동의한 수 분포:")
print(test["agree"].value_counts().sort_index(ascending=False).to_string())

# 불확실 = 모델 과반이 동의하지 않거나 확신 차이가 작음
test["unsure"] = (test["agree"] <= NM // 2) | (test["margin"] < 0.25)      # 과반 미동의 or 1·2위 확률 차이 < 0.25
n_u = int(test["unsure"].sum())
print(f"\n불확실 문항 {n_u}개 ({n_u / len(test):.1%})  — 캐글 오답 추정 약 {round(len(test) * (1 - 0.96455))}개")
print("확신 구간별 문항 수:"); print(pd.cut(test["conf"], [0, .4, .5, .6, .7, .8, .9, .95, 1.0]).value_counts().sort_index().to_string())

# 3. 질문 유형별 약점

In [ ]:
# 질문 유형 (키워드) — 불확실 비율이 평균보다 높은 유형이 약점
RULES = [
    ("전화번호", r"전화|연락처|번호.*자리|휴대폰|핸드폰"),
    ("가격·금액", r"가격|원\b|금액|얼마|요금|할인"),
    ("시간·날짜", r"시간|몇 시|날짜|요일|기간|영업|오픈|마감|월 |일 "),
    ("개수 세기", r"몇 개|몇 명|개수|몇 가지|몇 층|몇 번째"),
    ("위치·방향", r"왼쪽|오른쪽|위쪽|아래|가운데|옆|방향|위치|몇 층"),
    ("색상", r"색|컬러"),
    ("이름·상호", r"상호|이름|브랜드|가게|회사|제목|명칭"),
    ("영문", r"[A-Za-z]{3,}"),
    ("숫자 일반", r"\d|숫자|번호"),
]
def qtype(q):
    return [n for n, p in RULES if re.search(p, str(q))] or ["기타"]
test["types"] = test["question"].apply(qtype)
ex = test.explode("types")
t = ex.groupby("types").agg(문항수=("unsure", "size"), 불확실=("unsure", "sum"))
t["불확실비율"] = (t["불확실"] / t["문항수"]).round(3)
base = test["unsure"].mean()
t["평균대비"] = (t["불확실비율"] / base).round(2)
print(f"전체 불확실 비율 {base:.3f}")
display(t.sort_values("평균대비", ascending=False))

# 4. 원본 이미지 크기와의 관계

In [ ]:
# 원본 이미지 크기 — 불확실 문항이 큰 사진(=글자가 상대적으로 작음)에 몰려 있으면 '더 높은 해상도' 가 답
def img_size(rel):
    with Z.open(member(rel)) as f:
        im = Image.open(f); return im.size
sz = test["path"].apply(img_size)
test["W"] = [w for w, h in sz]; test["H"] = [h for w, h in sz]
test["MP"] = test["W"] * test["H"] / 1e6
test["long"] = test[["W", "H"]].max(1)
g = test.groupby("unsure")[["MP", "long"]].median().rename(index={False: "확신", True: "불확실"})
print("원본 이미지 중앙값 (메가픽셀 · 긴 변 px)"); display(g)
bins = pd.qcut(test["MP"], 5)
print("\n이미지 크기 5분위별 불확실 비율 (위로 갈수록 큰 사진)")
display(test.groupby(bins, observed=True)["unsure"].agg(["size", "mean"]).rename(columns={"size": "문항수", "mean": "불확실비율"}).iloc[::-1])

# 5. 가장 불확실한 16문항 직접 보기

In [ ]:
# 가장 불확실한 문항 직접 보기 (확신 차이가 작은 순)
def thumb(rel, w=420):
    with Z.open(member(rel)) as f:
        im = Image.open(f).convert("RGB")
    im.thumbnail((w, w)); b = io.BytesIO(); im.save(b, "JPEG", quality=80)
    return base64.b64encode(b.getvalue()).decode()
cols = list(P)
top = test.sort_values("margin").head(16)
html = []
for _, r in top.iterrows():
    i = r.name
    picks = " · ".join(f"{k}:{L[P[k][i].argmax()]}" for k in cols)
    opts = "<br>".join(f"<b>{x}</b>) {r[x]}" + (" ◀" if x == r['pred'] else "") for x in "abcd")
    html.append(f"""<div style="display:flex;gap:14px;margin:10px 0;padding:8px;border:1px solid #ddd;border-radius:8px">
      <img src="data:image/jpeg;base64,{thumb(r['path'])}" style="max-width:420px">
      <div style="font-size:13px;line-height:1.5"><b>{r['id']}</b> · {r['W']}×{r['H']} · 확신 {r['conf']:.2f} · 차이 {r['margin']:.2f}<br>
      <b>Q.</b> {r['question']}<br>{opts}<br><span style="color:#888">{picks}</span></div></div>""")
display(HTML("".join(html)))

# 6. 저장

In [ ]:
out = MY / "challenge" / "output" / "analysis"; out.mkdir(parents=True, exist_ok=True)
test.drop(columns=["types"]).to_csv(out / "test_uncertainty.csv", index=False)
print("저장:", out / "test_uncertainty.csv", "| 불확실 문항 id 목록은 unsure 열")

### 결과 읽는 법 → 다음 수
| 발견 | 의미 | 다음 수 |
|---|---|---|
| 불확실 문항이 **큰 사진**에 몰림 | 1024px 로 줄이면서 글자가 뭉개짐 | **불확실 문항만** 1280 · 1536px 로 다시 풀기 (수백 문항이라 싸다) |
| 특정 **질문 유형**(전화번호 · 개수 등)에 몰림 | 그 유형의 읽기 · 세기 능력 부족 | 그 유형만 해상도 · 크롭 TTA |
| 사진을 봐도 **사람도 헷갈림** | 진짜 애매한 문항 | 여기가 천장 → 마무리 |

# 7. 최종 보정 — 불확실 문항에서 한 모델의 과신 제거 (0.96455 → 0.96574)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")                       # 위 셀들이 드라이브를 해제했을 수 있어 다시 연결
from collections import Counter
sample = pd.read_csv(io.BytesIO(Z.read(pick("sample_submission.csv"))))
SUB = MY / "challenge" / "output" / "submissions"; SUB.mkdir(parents=True, exist_ok=True)
def save(tag, ans):
    s = sample.copy()
    s[sample.columns[1]] = s[sample.columns[0]].map(dict(zip(test["id"], ans)))
    assert s[sample.columns[1]].isna().sum() == 0
    s.to_csv(SUB / f"submission_{tag}.csv", index=False)
    print(f"[{tag}] m2(0.96455) 와 다른 답 {int(np.sum(ans != test['pred'].values))}개")

# (1) 불확실 문항만 6개 모델 다수결                     -> 공개 0.96514
def plurality(i):
    c = Counter(votes[i]).most_common()
    return ens[i].argmax() if len(c) > 1 and c[0][1] == c[1][1] else c[0][0]
pl = np.array([plurality(i) for i in range(len(test))])
save("vote_unsure", np.where(test["unsure"], L[pl], test["pred"]))

# (2) 6개 모델 같은 비중 확률 평균 — 불확실 문항만       -> 공개 0.96574
eq = sum(P[k] for k in P) / len(P)
save("vote_eqprob", np.where(test["unsure"], L[eq.argmax(1)], test["pred"]))

# (3) 6개 모델 같은 비중 확률 평균 — 전체 (최종 선택)    -> 공개 0.96574
save("eq_all", L[eq.argmax(1)])
drive.flush_and_unmount()